# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Missesqueensy/FlyRank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""One row represents one content page for one client during one reporting period.

For this assignment, I use a mid-panel month (2026-03) to avoid using the final month for feature development. My goal is to build features only from historical information that would have been available at the decision moment."""

'One row represents one content page for one client during one reporting period.\n\nFor this assignment, I use a mid-panel month (2026-03) to avoid using the final month for feature development. My goal is to build features only from historical information that would have been available at the decision moment.'

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ctr
- content_age_days

These variables are historical measurements that are available before making a prediction.

### Label (proxy)

Future content trend (Growth, Stable, or Decline) derived from future search performance.

### Context

- client_hash_id
- content_hash_id
- report_date
- content_type

These variables identify the content and provide context but are not prediction targets.

### Excluded

I exclude any future information such as future clicks, future impressions, future CTR, trend labels, or variables calculated from the prediction period because they would introduce data leakage."""


'### Features\n- gsc_impressions\n- gsc_clicks\n- gsc_avg_position\n- ctr\n- content_age_days\n\nThese variables are historical measurements that are available before making a prediction.\n\n### Label (proxy)\n\nFuture content trend (Growth, Stable, or Decline) derived from future search performance.\n\n### Context\n\n- client_hash_id\n- content_hash_id\n- report_date\n- content_type\n\nThese variables identify the content and provide context but are not prediction targets.\n\n### Excluded\n\nI exclude any future information such as future clicks, future impressions, future CTR, trend labels, or variables calculated from the prediction period because they would introduce data leakage.'

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""I will verify:

1. One row represents one content page for one client on one reporting date.
2. The number of rows and the available date range for the selected month.
3. The number of rows where GSC data is available using `gsc_data_available IS TRUE`."""

'I will verify:\n\n1. One row represents one content page for one client on one reporting date.\n2. The number of rows and the available date range for the selected month.\n3. The number of rows where GSC data is available using `gsc_data_available IS TRUE`.'

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""This dataset cannot explain or prove how Google's ranking algorithm works.

It only contains observed historical search performance data. Any relationships identified by the model should be interpreted as decision-support rather than causal conclusions.

Some clients have incomplete Google Search Console or Google Analytics coverage, which may limit the available observations."""

"This dataset cannot explain or prove how Google's ranking algorithm works.\n\nIt only contains observed historical search performance data. Any relationships identified by the model should be interpreted as decision-support rather than causal conclusions.\n\nSome clients have incomplete Google Search Console or Google Analytics coverage, which may limit the available observations."

In [5]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [6]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [7]:
con.sql(f"""
SELECT COUNT(*)
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()

,count_star()
0,9841378


In [8]:
print(TABLES)

{'dim_clients': "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')", 'dim_content': "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')", 'fact_daily': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')", 'fact_daily_sample': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')", 'fact_query_90d': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')"}


In [9]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


In [15]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30,
               SUM(f.gsc_clicks)/NULLIF(SUM(f.gsc_impressions),0) AS ctr_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,ctr_last30
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789,0.008232
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636,0.011735
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167,0.000000
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367,0.002037
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100,0.004334


In [13]:
con.sql(f"""
DESCRIBE
SELECT *
FROM {TABLES['fact_daily']}
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [11]:
# Query 1: number of rows + date range for March 2026
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [12]:
# Query 2: available GSC rows
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
  AND gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


In [14]:
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS n
FROM {TABLES['fact_daily']}
WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


In [ ]:
"""## Deliberate leakage experiment

To understand data leakage, I deliberately add one feature that contains information from the prediction target.

This should produce an unrealistically high score because the model is indirectly given the answer.

After observing the effect, I remove the leaked feature and keep only features that would have been available at the decision moment."""

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np

# Copy the feature table
df = features.copy()

# Simple label for demonstration
df["label"] = (df["imp_last30"] > df["imp_prev30"]).astype(int)

# -----------------------------
# Honest model
# -----------------------------
X = df[["imp_prev30",
        "imp_last30",
        "clk_last30",
        "pos_last30",
        "ctr_last30"]]

y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Honest accuracy:", round(accuracy_score(y_test, pred),3))

# -----------------------------
# Deliberate leakage
# -----------------------------
X_leak = X.copy()

# BAD FEATURE (contains the answer)
X_leak["leaked_label"] = y

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy WITH leakage:", round(accuracy_score(y_test, pred),3))

Honest accuracy: 0.993
Accuracy WITH leakage: 1.0


In [ ]:
### Leakage conclusion

"""The honest model achieved an accuracy of 0.993.

After deliberately adding a label-derived feature, the accuracy increased to 1.000, demonstrating how even a small amount of leaked information can produce overly optimistic results.

Although the increase is small in this example, it shows that leakage inflates evaluation metrics and should always be avoided. The leaked feature was removed, and only historical features available at the decision moment should be used in the final model."""

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.